In [24]:
import pandas as pd
import glob
import subprocess
import os
import time
import psutil
import time
from typing import NamedTuple

## Notebook Utils

In [25]:
METADATA = pd.read_csv('metadata.csv', index_col="Analyte ID")
METADATA

,kit
Analyte ID,
999999,TAPS


In [26]:
class TVCResult(NamedTuple):
    """Container for storing TVC  results."""
    sample_id: str
    runtime_minutes: float
    memory_mb: float
    cores_used: int
    vcf_path: str

In [27]:
class BenchmarkResult(NamedTuple):
    """Container for storing benchmarking results."""
    sample_id: str
    snp_precision: float
    snp_recall: float
    snp_f1: float
    indel_precision: float
    indel_recall: float
    indel_f1: float

In [28]:
class SampleMetadata(NamedTuple):
    """Container for storing sample metadata."""
    sample_id: str
    median_depth_of_coverage: int
    kit: str

In [29]:

# VCF and Benchmark regions from https://www.nist.gov/programs-projects/genome-bottle version v4.2.1
BENCHMARK_VCF = "HG001_GRCh38_1_22_v4.2.1_benchmark.vcf.gz"
BENCHMARK_REGIONS = "HG001_GRCh38_1_22_v4.2.1_benchmark.bed"
REFERENCE_GENOME = "hg38+pUC19+lambda+plasmid.fa"

In [30]:
def run_TVC(input_bam, input_ref, cores):
    """
    Run TVC variant caller and monitor resource usage across all threads/processes.
    
    :param input_bam: Path to input BAM file.
    :param input_ref: Path to reference genome file.
    :param cores: Number of CPU cores to use.
    :return: TVCResult containing runtime and memory usage information.
    """
    output_vcf = input_bam.replace(".bam", ".tvc.vcf")
    if os.path.exists(output_vcf):
        return TVCResult(
            sample_id=input_bam.split(".")[0],
            runtime_minutes=0.0,
            memory_mb=0.0,
            vcf_path=output_vcf,
            cores_used=cores
        )
    cmd = f"../target/release/tvc -t {cores} {input_ref} {input_bam} {output_vcf}"

    start_time = time.time()
    proc = subprocess.Popen(cmd, shell=True)
    ps_proc = psutil.Process(proc.pid)

    peak_memory = 0
    while proc.poll() is None:
        try:
            # Include child processes’ memory
            mem_total = ps_proc.memory_info().rss
            for child in ps_proc.children(recursive=True):
                mem_total += child.memory_info().rss
            peak_memory = max(peak_memory, mem_total)
        except psutil.NoSuchProcess:
            break
        time.sleep(0.05)

    end_time = time.time()
    elapsed_minutes = (end_time - start_time) / 60

    result = TVCResult(
        sample_id=input_bam.split(".")[0],
        runtime_minutes=elapsed_minutes,
        memory_mb=peak_memory / (1024 ** 2),
        vcf_path=output_vcf,
        cores_used=cores
    )
    return result


In [31]:
def extract_accuracy_metrics(benchmark_vcf, test_vcf, regions_bed, reference_genome):
    """
    Extract accuracy metrics using hap.py Docker container.
    :param benchmark_vcf: Path to benchmark VCF file.
    :param test_vcf: Path to test VCF file.
    :param regions_bed: Path to BED file defining regions of interest.
    :param reference_genome: Path to reference genome file.
    :return: BenchmarkResult containing precision, recall, and F1 scores for SNPs and INDELs.
    """
    output_prefix = f"{os.path.splitext(test_vcf)[0]}.happy"
    metrics_file = f"{output_prefix}.summary.csv"
    docker_image = "quay.io/biocontainers/hap.py:0.3.14--py27h5c5a3ab_0"
    if not os.path.exists(metrics_file):
        cmd = f"sudo docker run --rm -v {os.getcwd()}:/data {docker_image} hap.py /data/{benchmark_vcf} /data/{test_vcf} -r /data/{reference_genome} -f /data/{regions_bed} -o /data/{output_prefix} --threads 90"
        subprocess.run(cmd, shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)

    df = pd.read_csv(metrics_file)
    for row in df.itertuples():
        if row.Filter != 'PASS':
            continue
        if row.Type == 'SNP':
            snp_precision = row._12
            snp_recall = row._11
            snp_f1 = row._14
        elif row.Type == 'INDEL':
            indel_precision = row._12
            indel_recall = row._11
            indel_f1 = row._14

    
    result = BenchmarkResult(
        sample_id=test_vcf.split(".")[0],
        snp_precision=snp_precision,
        snp_recall=snp_recall,
        snp_f1=snp_f1,
        indel_precision=indel_precision,
        indel_recall=indel_recall,
        indel_f1=indel_f1
    )
    return result

In [32]:
def extract_sample_metadata(analyte, metadata):
    """
    Extract sample metadata including median depth of coverage, input mass, and kit type.
    :param analyte: Analyte ID of the sample.
    :param metadata: DataFrame containing metadata information.
    :return: SampleMetadata containing sample metadata.
    """

    row = metadata.loc[int(analyte)]
    kit = row['kit']

    wgs_data_csv = pd.read_csv(f'{analyte}.CollectWgsMetrics.coverage_metrics', sep='\t', comment='#', nrows=1)
    depth_of_coverage = int(wgs_data_csv['MEDIAN_COVERAGE'])
    metadata = SampleMetadata(
        sample_id=analyte,
        median_depth_of_coverage=depth_of_coverage,
        kit=kit
    )
    return metadata

## Analysis on Taps and Unconverted

In [ ]:
input_bams = glob.glob("*.bam")
tvc_results = []
accuracy_results = []
metadata_results = []
for bam in input_bams:
    tvc_result = run_TVC(bam, REFERENCE_GENOME, cores=90)
    accuracy_results.append(extract_accuracy_metrics(BENCHMARK_VCF, tvc_result.vcf_path, BENCHMARK_REGIONS, REFERENCE_GENOME))
    metadata_results.append(extract_sample_metadata(os.path.basename(bam.split(".")[0]), METADATA))
    tvc_results.append(tvc_result)


In [ ]:
tvc_df = pd.DataFrame(tvc_results)
tvc_df = tvc_df.set_index('sample_id')
accuracy_df = pd.DataFrame(accuracy_results)
accuracy_df = accuracy_df.set_index('sample_id')
metadata_df = pd.DataFrame(metadata_results)
metadata_df = metadata_df.set_index('sample_id')

final_df = pd.concat([tvc_df, accuracy_df, metadata_df], axis=1)
final_df.to_csv("tvc_benchmarking_results.tsv", sep="\t")

## Results on demo data

In [35]:
demo_output_file = "demo_tvc_benchmarking_results.tsv"
if os.path.exists(demo_output_file):
    demo_final_df = pd.read_csv(demo_output_file, sep="\t")
else:
    demo_tvc_results = run_TVC("999999.bam", REFERENCE_GENOME, cores=90)
    demo_happy_results = extract_accuracy_metrics(BENCHMARK_VCF, demo_tvc_results.vcf_path, BENCHMARK_REGIONS, REFERENCE_GENOME)
    demo_metadata = extract_sample_metadata("999999", METADATA)

    demo_tvc_df = pd.DataFrame([demo_tvc_results])
    demo_happy_df = pd.DataFrame([demo_happy_results])
    demo_metadata_df = pd.DataFrame([demo_metadata])

    demo_final_df = pd.concat([demo_tvc_df, demo_happy_df, demo_metadata_df], axis=1)
    demo_final_df.to_csv(demo_output_file, sep="\t", index=False)
    
print(demo_final_df)

2026-08-13T14:44:20.800304Z  INFO Starting TVC workflow
2026-08-13T14:44:20.801355Z  INFO Reading reference sequences
2026-08-13T14:44:38.565360Z  INFO Dividing genome into chunks and getting ready for parallel processing
  sample_id  runtime_minutes    memory_mb  cores_used        vcf_path  \
0    999999        44.284119  6975.765625          90  999999.tvc.vcf   

  sample_id  snp_precision  snp_recall   snp_f1  indel_precision  \
0    999999       0.979364    0.942773  0.96072         0.836968   

   indel_recall  indel_f1 sample_id  median_depth_of_coverage   kit  
0       0.82399  0.830428    999999                        40  TAPS  


/tmp/ipykernel_4803/2730702165.py:13: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  depth_of_coverage = int(wgs_data_csv['MEDIAN_COVERAGE'])
